# 2. Put the material into native FEM

**Goal:** turn lesson 1's density into mesh energy, its displacement gradient,
and stiffness. We read tetrahedral and hexahedral meshes of the bunny with
`mesh.read_veg` and assemble both at the same prescribed affine displacement.

The stress $\partial W/\partial F$ describes one material point. The gradient
$g=\partial E/\partial u$ describes how mesh energy changes when a vertex moves.
Native FEM connects the two through the displacement basis and reference-volume
quadrature.

In [ ]:
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
from litepgo import dof, energy, fem, mesh
from litepgo.torch import TorchVolumetricMaterial

torch.set_num_threads(1)
dtype = torch.float64

## One small adapter around the same law

Subclass `TorchVolumetricMaterial` and provide `num_parameters` and `W`.
Keep the multiplicative split and $\det F_p$ factor from lesson 1.
This mesh example uses $F_p=I$ at every point.

`W` receives `(N,3,3)`, `(N,3,3)`, `(N,2)` and returns `(N,)`. The base class computes
requested derivatives automatically. Invalid physical parameters return NaN so native
FEM can reject the evaluation.

In [ ]:
class TorchNeoHookean(TorchVolumetricMaterial):
    num_parameters = 2

    @staticmethod
    def W(F, Fp, theta):
        Fe = F @ torch.linalg.inv(Fp)
        lam, mu = theta.unbind(-1)
        log_J = torch.linalg.det(Fe).log()
        psi = 0.5 * mu * (Fe.square().sum((-2, -1)) - 3) - mu * log_J + 0.5 * lam * log_J.square()
        W = torch.linalg.det(Fp) * psi
        return torch.where((lam >= 0) & (mu > 0), W, torch.full_like(W, float('nan')))

## Select material, displacement basis, and quadrature

`TetP1Basis` uses linear displacement interpolation and one centroid sample per
tetrahedron. `HexQ1Basis` uses trilinear interpolation and a $2\times2\times2$
Gauss rule, which gives eight samples per hexahedron. Both use the same Torch
density, constant $F_p=I$, and the same material parameters.

Quadrature weights convert $W$ to energy per element. The basis maps $P$ and
$\partial P/\partial F$ back to nodal contributions, and the DOF map adds the
contributions from elements that share a vertex.

In [ ]:
asset_dir = next((folder for folder in (Path("."), Path("assets"), Path("../../assets"))
                  if (folder / "bunny-tet.veg").is_file()), Path("."))

def make_problem(filename, basis, quadrature, theta):
    volume = mesh.read_veg(asset_dir / filename)
    X, topology = volume.positions, volume.topology
    mapping = dof.VertexDofMap(topology, 3)
    rest = X.ravel()
    geometry = [fem.InterpolatedResponse[basis](mapping.gather(e, rest))
                for e in range(topology.num_cells)]
    Fp_field = energy.AffineResponse(np.zeros((9, 3)), np.eye(3).ravel(order='F'))
    theta_field = energy.AffineResponse(np.zeros((2, 3)), theta)
    plastic = [Fp_field] * topology.num_cells
    fields = [theta_field] * topology.num_cells
    torch_energy = fem.VolumetricFemEnergy[TorchNeoHookean, basis, quadrature](
        mapping, geometry, plastic, fields)
    native_energy = fem.VolumetricFemEnergy[fem.NeoHookean, basis, quadrature](
        mapping, geometry, plastic, fields)
    return volume, torch_energy, native_energy

strain = np.array([[.32, .12, 0], [0, -.08, .08], [.04, 0, .16]])
theta = np.array([2., 3.])
mesh_cases = {
    'Tet bunny': ('bunny-tet.veg', fem.TetP1Basis, fem.TetCentroidQuadrature),
    'Hex bunny': ('bunny-hex.veg', fem.HexQ1Basis, fem.GaussLegendre3D[2]),
}
problems = {label: make_problem(*case, theta) for label, case in mesh_cases.items()}

## Compare with native Neo-Hookean

For each mesh, change only the material in the template key. The Torch and native
versions must give the same energy $E$, positive energy gradient $g$, and exact
stiffness $K$. Compare $E$ and every component of $g$ at rest ($u=0$) and
at the prescribed displacement. The output shows the DOF count, both gradient
norms, and the three displacement and gradient components at the vertex with
the largest Torch gradient norm in each state. $E$ is one scalar for the whole
mesh, and $g$ has one entry per displacement DOF.

At the prescribed displacement we also check the sparse matrix difference
and verify $K\,\Delta u$ against the change in gradient along $\Delta u$.

The tet and hex meshes have different discretizations and each has its own
native reference. This prescribed-displacement example leaves rigid translation
modes in the stiffness matrix.

In [ ]:
def compare_energy_gradient(label, u, E, g, En, gn):
    np.testing.assert_allclose(E, En, rtol=1e-11, atol=1e-13)
    np.testing.assert_allclose(g, gn, rtol=1e-10, atol=1e-12)
    vertex = np.linalg.norm(g.reshape(-1, 3), axis=1).argmax()
    xyz = slice(3 * vertex, 3 * vertex + 3)
    print(f'{label} · {u.size:,} displacement DOFs')
    print(f'  Torch   E = {E:.7e}   ||g|| = {np.linalg.norm(g):.7e}')
    print(f'  Native  E = {En:.7e}   ||g|| = {np.linalg.norm(gn):.7e}')
    print(f'  Max gradient error: {np.max(np.abs(g - gn)):.2e}')
    print(f'  Vertex {vertex}: u = {u[xyz]}')
    print(f'    Torch g = {g[xyz]}   Native g = {gn[xyz]}')

def check_assembly(label, problem, strain):
    volume, potential, native = problem
    X = volume.positions
    u = (X @ strain.T).ravel()
    E, g, K = potential.eval(u, gradient=True, hessian=energy.HessianKind.Exact)
    En, gn, Kn = native.eval(u, gradient=True, hessian=energy.HessianKind.Exact)
    compare_energy_gradient(f'{label} · Deformed', u, E, g, En, gn)
    stiffness_error = np.max(np.abs((K - Kn).data), initial=0.)
    assert stiffness_error < 1e-11

    direction = np.linspace(-.2, .3, len(u))
    h = 1e-6
    fd = (potential.eval_gradient(u + h * direction)
          - potential.eval_gradient(u - h * direction)) / (2 * h)
    np.testing.assert_allclose(K @ direction, fd, rtol=1e-6, atol=1e-9)
    print(f'{label}: {len(X):,} vertices, {volume.topology.num_cells:,} elements')
    print(f'K: {K.shape}, nonzeros: {K.nnz:,}')
    print(f'Max stiffness error: {stiffness_error:.2e}. Directional check passed.')
    return E, g, K

baseline = {}
for label, problem in problems.items():
    volume, potential, native = problem
    rest_u = np.zeros(volume.positions.size)
    E, g, _ = potential.eval(rest_u, gradient=True)
    En, gn, _ = native.eval(rest_u, gradient=True)
    compare_energy_gradient(f'{label} · Rest', rest_u, E, g, En, gn)
    np.testing.assert_allclose([E, En], 0., atol=1e-13)
    np.testing.assert_allclose(g, 0., atol=1e-12)
    np.testing.assert_allclose(gn, 0., atol=1e-12)
    baseline[label] = check_assembly(label, problem, strain)
    print()

## See the prescribed deformation

Each row shows the same mesh at rest ($X$) and at the prescribed deformation
($X+u$) used by the FEM calculation. Both views share the camera and scale.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7), subplot_kw={'projection': '3d'},
                         constrained_layout=True)
for row, (label, (volume, _, _)) in zip(axes, problems.items()):
    surface = mesh.extract_surface_with_vertices(volume)
    faces = surface.mesh.cells
    surface_nodes = volume.positions[np.asarray(surface.vertices)]
    X = surface_nodes[:, [0, 2, 1]]
    displacement = (surface_nodes @ strain.T)[:, [0, 2, 1]]
    views = (X, X + displacement)
    bounds = np.vstack(views)
    lower, upper = bounds.min(axis=0), bounds.max(axis=0)
    for ax, positions, state, color in zip(
        row, views, ('Rest', 'Prescribed'), ('silver', 'cornflowerblue'),
    ):
        ax.plot_trisurf(*positions.T, triangles=faces, color=color, linewidth=0)
        ax.set(xlim=(lower[0], upper[0]), ylim=(lower[1], upper[1]),
               zlim=(lower[2], upper[2]), title=f'{label} · {state}')
        ax.set_box_aspect(upper - lower)
        ax.set_proj_type('ortho')
        ax.view_init(elev=15, azim=-65)
        ax.set_axis_off()
plt.show()

## Try it

1. Double the off-diagonal entries `strain[0, 1]` and `strain[1, 2]`. For both
   meshes, repeat the native comparison and the stiffness directional check.
2. Restore the original displacement and double both material parameters.
   Which of $E$, $g$, and $K$ change? Does the stiffness sparsity pattern change?

Both checks still pass at the new shear. The off-diagonal changes produce a
different deformation gradient, so the energy values change.

At the original pose, doubling $\lambda$ and $\mu$ doubles $E$, $g$, and $K$.
The density is linear in the two material parameters, and taking derivatives
with respect to displacement preserves that factor. The mesh connectivity
and stiffness sparsity pattern stay the same.

The chart divides each output magnitude by its original value on the same
mesh. The dashed line marks the original value. For the gradient and stiffness,
the chart uses the vector norm $\|g\|_2$ and the matrix norm $\|K\|_F$.

In [ ]:
def magnitudes(E, g, K):
    return np.array([E, np.linalg.norm(g), np.linalg.norm(K.data)])

sheared = strain.copy()
sheared[0, 1] *= 2
sheared[1, 2] *= 2

shear_ratios = {}
print('Increased shear')
for label, problem in problems.items():
    shear_E, shear_g, shear_K = check_assembly(label, problem, sheared)
    shear_ratios[label] = (magnitudes(shear_E, shear_g, shear_K)
                           / magnitudes(*baseline[label]))
    print(f'Original energy {baseline[label][0]:.7g} → {shear_E:.7g}')

doubled_ratios = {}
print('\nDoubled material parameters at the original pose')
for label, case in mesh_cases.items():
    volume, potential, _ = make_problem(*case, 2 * theta)
    u = (volume.positions @ strain.T).ravel()
    E, g, K = baseline[label]
    doubled_E, doubled_g, doubled_K = potential.eval(
        u, gradient=True, hessian=energy.HessianKind.Exact)
    np.testing.assert_allclose(doubled_E, 2 * E, rtol=1e-11, atol=1e-13)
    np.testing.assert_allclose(doubled_g, 2 * g, rtol=1e-10, atol=1e-12)
    assert np.max(np.abs((doubled_K - 2 * K).data), initial=0.) < 1e-11
    np.testing.assert_array_equal(doubled_K.indptr, K.indptr)
    np.testing.assert_array_equal(doubled_K.indices, K.indices)
    doubled_ratios[label] = (magnitudes(doubled_E, doubled_g, doubled_K)
                             / magnitudes(E, g, K))
    print(f'{label}: E {E:.7g} → {doubled_E:.7g}. E, g, and K doubled. Same sparse pattern.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True, constrained_layout=True)
x = np.arange(3)
for ax, label in zip(axes, problems):
    shear_bars = ax.bar(x - .18, shear_ratios[label], width=.36, label='Increased shear')
    doubled_bars = ax.bar(x + .18, doubled_ratios[label], width=.36, label='Doubled material parameters')
    ax.axhline(1, color='gray', linestyle='--', linewidth=1, label='Original')
    ax.bar_label(shear_bars, fmt='%.3f×', padding=3, fontsize=8)
    ax.bar_label(doubled_bars, fmt='%.3f×', padding=3, fontsize=8)
    ax.set_xticks(x, [r'$E$', r'$\|g\|_2$', r'$\|K\|_F$'])
    ax.set_title(label)
    ax.margins(y=.15)
axes[0].set_ylabel('Ratio to original')
fig.legend(*axes[0].get_legend_handles_labels(), loc='outside upper center', ncol=3)
plt.show()

**Checkpoint** Torch defines the density. Native FEM handles geometry, quadrature,
chain rules, and assembly. Next, fit material parameters at a fixed pose.